In [1]:
puts `head -3 ./raw_data/bv-kg-20250225.large`

source_1	id_1	type_1	name_1	source_2	id_2	type_2	name_2	score	url
UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	HP:human_phenotype	HP:0001263	Human Phenotype	Global developmental delay	0.0501869	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Human%20Phenotype%7CGlobal%20developmental%20delay%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D
UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	HP:human_phenotype	HP:0001249	Human Phenotype	Intellectual disability	0.0438494	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Human%20Phenotype%7CIntellectual%20disability%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D


In [2]:
puts `grep Drug ./raw_data/bv-kg-20250225.large | head -3 `


UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	MeSH	D005680	Drug	gamma-Aminobutyric Acid	0.00591104	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Drug%7Cgamma-Aminobutyric%20Acid%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D
UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	MeSH	D012978	Drug	Sodium Oxybate	0.00210581	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Drug%7CSodium%20Oxybate%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D
UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	MeSH	D020888	Drug	Vigabatrin	0.0009236	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Drug%7CVigabatrin%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D


In [3]:
require 'csv'
drugs = {}
sources = {}
CSV.foreach('./raw_data/bv-kg-20250225.large', col_sep: "\t", quote_char: '"', liberal_parsing: true, headers: true) do |row|
  if row["type_1"] == "Drug"
    sources[row["source_1"]] = 1
  end
  if row["type_2"] == "Drug"
    sources[row["source_2"]] = 1
  end
end

puts sources.keys


MeSH
FDA Drugs


# FAILs
FDA 4901d7c1b4e2aef6913d880bfc91240e
Mathces nothing by google or Grok.

All others are MeSH

# Map MeSH to PubChem CUI and formal name

In [4]:
require 'json'
require 'rest-client'

true

In [5]:
meshdrugs = {}
CSV.foreach('./raw_data/bv-kg-20250225.large', col_sep: "\t", quote_char: '"', liberal_parsing: true, headers: true) do |row|
  # this is to eliminate duplicates
  if row["type_1"] == "Drug"
    meshdrugs[row["id_1"]] = row["name_1"]
  end
  if row["type_2"] == "Drug"
    meshdrugs[row["id_2"]] = row["name_2"]
  end
end

puts meshdrugs.keys.size
puts "examples"
puts meshdrugs.keys[0..4]



477
examples
D005680
D012978
D020888
C066471
D014635


In [6]:

require 'rest-client'
require 'json'
RATE_LIMIT_SLEEP = 0.34 # ~3 req/sec safe without API key
REQUEST_TIMEOUT = 30    # seconds
URI_PARSER = URI::Parser.new


def fetch_mesh_json(mesh_ui)
  mesh_ui = mesh_ui.strip.upcase
  url = "https://id.nlm.nih.gov/mesh/#{mesh_ui}.json"
  
  begin
    response = RestClient::Request.execute(
      method: :get,
      url: url,
      timeout: REQUEST_TIMEOUT
    )
    
    if response.code == 200
      JSON.parse(response.body)
    else
      warn "Error fetching MeSH #{mesh_ui}: HTTP #{response.code}"
      false
    end
  rescue RestClient::ExceptionWithResponse => e
    puts "MeSH #{mesh_ui} HTTP error: #{e.message} (code #{e.response&.code})"
    false
  rescue => e
    puts "Exception fetching MeSH #{mesh_ui}: #{e.message}"
    false
  end
end

def fetch_pubchem_cids(endpoint, identifier)
  escaped = URI_PARSER.escape(identifier.strip)
  url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/#{endpoint}/#{escaped}/cids/JSON"
  
  begin
    response = RestClient::Request.execute(
      method: :get,
      url: url,
      timeout: REQUEST_TIMEOUT
    )
    
    if response.code == 200
      data = JSON.parse(response.body)
      cids = data.dig('IdentifierList', 'CID') || []
      cids
    else
      # Non-200 (e.g., 404 for no match) treated as no results, not fatal error
      []
    end
  rescue RestClient::NotFound
    [] # Explicit 404 = no match
  rescue RestClient::ExceptionWithResponse => e
    puts "PubChem #{endpoint} #{identifier} HTTP error: #{e.message} (code #{e.response&.code})"
    false
  rescue => e
    puts "Exception fetching PubChem #{endpoint} #{identifier}: #{e.message}"
    false
  end
end

def extract_cas_like_rn(json)
  rn = json['registryNumber']&.strip
  return nil if rn.nil? || rn.empty? || rn == '0'
  
  # Strict CAS format check (ignores UNII/EC)
  if rn =~ /\A\d{1,7}-\d{2}-\d\z/
    rn
  else
    nil
  end
end


# Example
# Example
# warn fetch_mesh_json('D020888')
# j = fetch_mesh_json('D020888')
# warn fetch_mesh_json('D012701'); abort
# j = fetch_mesh_json('D012701')

# puts map_mesh_to_cid('D012978')
# puts map_mesh_to_cid('D005680')

:extract_cas_like_rn

# Iteration over all mesh terms

In [7]:
STEREO_PREFIX = /\A(?:(?:[LlDd]-)|(?:\([RrSsEeZz\+\-RS]\)-)|(?:\(\+\/\-\)-))+/

def get_pubchem_title(cid)
  return nil unless cid.to_s =~ /^\d+$/
  url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/#{cid}/property/Title/JSON"
  resp = RestClient::Request.execute(method: :get, url: url, timeout: 15)
  return nil unless resp.code == 200
  JSON.parse(resp.body).dig('PropertyTable', 'Properties', 0, 'Title')
rescue
  nil
end

:get_pubchem_title

In [ ]:

CSVFILE = "./raw_data/bv-disease-graph.large".freeze
OUTPUT = "./maps/2025-biovista-drugs.map".freeze
ERROR  = "./maps/2025-biovista-drugs-errors".freeze
URI_PARSER = URI::Parser.new

out = File.open(OUTPUT, "w")
out.write CSV.generate_line(["biovista_meshid","biovista_label","CID","IUPACname"])

error = File.open(ERROR, "w")

meshdrugs.keys.each do |mesh|
    biovistaname = meshdrugs[mesh]
    json = fetch_mesh_json(mesh)
    if json == false
      warn "#{mesh} ERROR mesh_fetch_failed\n"
      error.write "#{mesh} ERROR mesh_fetch_failed\n"
      sleep(RATE_LIMIT_SLEEP)
      next
    end

# Robust extraction of preferred name (label can be string or hash)
    raw_label = json['label']
    if raw_label.is_a?(Hash)
      name = raw_label['@value'] || 'UNKNOWN'
    elsif raw_label.is_a?(String)
      name = raw_label
    else
      name = 'UNKNOWN'
    end
    name = name.strip

    cids = false
    method = 'none'
    cid_list = ''

    
    # 1. Try RegistryID (MeSH UI directly)
    result = fetch_pubchem_cids('xref/RegistryID', mesh)
    if result == false
      warn "Error on CID for #{mesh} - first attempt"
      error.write "#{mesh} ERROR cid fetch failed on attempt 1\n"
    elsif !result.empty?
      cids = result
      method = 'registry_id'
    end

    # 2. Try CAS if available and we don't have cids yet
    if cids == false || cids.empty?
      cas = extract_cas_like_rn(json)
      if cas
        result = fetch_pubchem_cids('xref/RN', cas)
        if result == false
          warn "Error on CID for #{mesh} - second attempt"
          error.write "#{mesh} ERROR cid fetch failed on attempt 2\n"
        elsif !result.empty?
          cids = result
          method = 'cas_rn'
        end
      end
    end

    # 3. Fallback to name search
    if (cids == false || cids.empty?) && name != 'UNKNOWN'
      result = fetch_pubchem_cids('name', name)
      if result == false
          warn "Error on CID for #{mesh} - third attempt"
          error.write "#{mesh} ERROR cid fetch failed on attempt 3\n"
      elsif !result.empty?
        cids = result
        method = 'name_search'
      end
    end

    if cids == false || cids.empty?
      warn  "No PubChem CID found for #{mesh} (#{name})"
      error.write "No PubChem CID found for #{mesh} (#{name})\n"
      next
    end

    cids.each do |cid|
        pubchem_title = get_pubchem_title(cid)
        iupac_label = pubchem_title ? pubchem_title.gsub(STEREO_PREFIX, '') : name
        out.write CSV.generate_line(["http://purl.bioontology.org/ontology/MESH/#{mesh}",biovistaname,cid,iupac_label])
        warn CSV.generate_line(["http://purl.bioontology.org/ontology/MESH/#{mesh}",biovistaname,cid,iupac_label])
    end
end

out.close
error.close

puts "DONE!"
  

    

(irb):4: warning: already initialized constant Object::URI_PARSER
(irb):5: warning: previous definition of URI_PARSER was here
http://purl.bioontology.org/ontology/MESH/D005680,gamma-Aminobutyric Acid,119,Gamma-Aminobutyric Acid
http://purl.bioontology.org/ontology/MESH/D012978,Sodium Oxybate,23663870,Sodium Oxybate
http://purl.bioontology.org/ontology/MESH/D020888,Vigabatrin,5665,Vigabatrin
http://purl.bioontology.org/ontology/MESH/C066471,NCS 382,3613485,Ncs 382
http://purl.bioontology.org/ontology/MESH/D014635,Valproic Acid,3121,Valproic Acid
http://purl.bioontology.org/ontology/MESH/D004298,Dopamine,681,Dopamine
http://purl.bioontology.org/ontology/MESH/D011736,Pyridoxine,1054,Pyridoxine
No PubChem CID found for D013482 (Superoxide Dismutase)
http://purl.bioontology.org/ontology/MESH/D005442,Flumazenil,3373,Flumazenil
http://purl.bioontology.org/ontology/MESH/C079148,(3-aminopropyl)(n-butyl)phosphinic acid,130021,(3-Aminopropyl)(n-butyl)phosphinic acid
http://purl.bioontology.org/o

MeSH 4901D7C1B4E2AEF6913D880BFC91240E HTTP error: 404 Not Found (code 404)


4901d7c1b4e2aef6913d880bfc91240e ERROR mesh_fetch_failed
http://purl.bioontology.org/ontology/MESH/D009536,Niacinamide,936,Nicotinamide
http://purl.bioontology.org/ontology/MESH/D014212,Tretinoin,444795,Retinoic Acid
http://purl.bioontology.org/ontology/MESH/D009555,Ninhydrin,10236,Ninhydrin
http://purl.bioontology.org/ontology/MESH/C541932,Ku 0063794,16736978,Ku-0063794
http://purl.bioontology.org/ontology/MESH/D004837,Epinephrine,5816,Epinephrine
http://purl.bioontology.org/ontology/MESH/C085075,CGP 55845A,9954841,Cgp 55845A
http://purl.bioontology.org/ontology/MESH/D003091,Colistin,5311054,Colistin
http://purl.bioontology.org/ontology/MESH/D011729,Pyridostigmine Bromide,7550,Pyridostigmine Bromide
http://purl.bioontology.org/ontology/MESH/D000109,Acetylcholine,187,Acetylcholine
http://purl.bioontology.org/ontology/MESH/D003345,Corticosterone,5753,Corticosterone
http://purl.bioontology.org/ontology/MESH/C004691,colistinmethanesulfonic acid,216258,Colistinmethanesulfonic acid
http://p